In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import xarray as xr
import earthaccess

def extract_transect(ds, var_name, target_lats, target_lons):
    """
    Extracts the nearest pixels to a defined transect line.
    """
    lat_vals = ds["latitude"].values
    lon_vals = ds["longitude"].values
    data_da = ds[var_name]

    rows, cols = [], []
    for lat, lon in zip(target_lats, target_lons):
        # Calculate Manhattan distance to find the nearest grid point
        dist = np.abs(lat_vals - lat) + np.abs(lon_vals - lon)
        i, j = np.unravel_index(dist.argmin(), lat_vals.shape)
        rows.append(i)
        cols.append(j)

    # Advanced indexing to pull points along the transect
    selection = data_da.isel(
        number_of_lines=xr.DataArray(rows, dims="points"), 
        pixels_per_line=xr.DataArray(cols, dims="points")
    ).load()
    
    # Assign actual geographic coordinates found to the new DataArray
    selection = selection.assign_coords(
        longitude=("points", [lon_vals[r, c] for r, c in zip(rows, cols)]),
        latitude=("points", [lat_vals[r, c] for r, c in zip(rows, cols)])
    )
    return selection

def plot_spectral_transect(ds_transect, filename, date_str, output_path):
    """
    Generates and saves the spectral variation plot.
    """
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    
    # Distance in degrees from the start of the transect for coloring
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    cmap = plt.get_cmap('plasma')
    
    fig, ax = plt.subplots(figsize=(12, 7))
    
    for i, p in enumerate(ds_transect.points):
        point_data = ds_transect.sel(points=p)
        color = cmap(norm(dist_degrees[i]))
        
        point_data.plot.line(
            ax=ax, 
            x='wavelength_3d', 
            marker='.', 
            color=color, 
            alpha=0.6, 
            add_legend=False
        )
    
    # Add colorbar for spatial context
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('Distance from start (Degrees °)', rotation=270, labelpad=15)
    
    ax.set_title(f"Spectral Signatures: Variation Along Transect\n{date_str}", fontsize=14)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Surface Reflectance (rhos)")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # Save the figure and close to free memory
    save_name = os.path.join(output_path, f"{filename}.png")
    plt.savefig(save_name, dpi=300)
    plt.close(fig)

def run_event_analysis(event_name, date_range, transect_lats, transect_lons, short_name='PACE_OCI_L2_SFREFL'):
    """
    Main workflow: search, download, process, and save plots for a specific event.
    """
    # Create output subdirectory
    output_dir = os.path.join("output_reports", event_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # Define search bounding box based on transect limits
    bbox = (min(transect_lons)-0.5, min(transect_lats)-0.5, 
            max(transect_lons)+0.5, max(transect_lats)+0.5)
    
    print(f"Searching data for event: {event_name}...")
    results = earthaccess.search_data(
        short_name = short_name,
        temporal = date_range,
        bounding_box = bbox
    )
    
    if not results:
        print("No data found for the given parameters.")
        return
    
    print(f"Found {len(results)} files. Opening fileset...")
    fileset = earthaccess.open(results)

    for file_obj in fileset:
        try:
            # Load data using DataTree for L2 structure
            dt = xr.open_datatree(file_obj, decode_timedelta=False, chunks={})
            ds = xr.merge(dt.to_dict().values())
            ds = ds.set_coords(("longitude", "latitude")).unify_chunks()
            
            # Extract data along the transect
            rhos_transect = extract_transect(ds, 'rhos', transect_lats, transect_lons)
            
            # Parsing file name for labeling
            # Format: PACE_OCI.YYYYMMDDTHHMMSS.L2.OC_CU.nc
            file_id = file_obj.full_name.split('.')[1]
            formatted_date = f"{file_id[:4]}-{file_id[4:6]}-{file_id[6:8]} {file_id[9:11]}:{file_id[11:13]}"
            
            # Plot and save
            plot_spectral_transect(rhos_transect, file_id, formatted_date, output_dir)
            print(f"Successfully processed: {file_id}")
            
        except Exception as e:
            print(f"Error processing file {file_obj.full_name}: {e}")



In [ ]:
# ---- EXECUTION EXAMPLE ----

if __name__ == "__main__":
    # Event Configuration
    EVENT_ID = "Malacca_Strait_Bloom"
    DATES = ('2025-06-12', '2025-06-30')
    
    # Define Transect Geometry
    NUM_POINTS = 50
    LATS = np.linspace(1.9, 2.5, NUM_POINTS)
    LONS = np.linspace(102, 101.4, NUM_POINTS)

    # Run Analysis
    run_event_analysis(EVENT_ID, DATES, LATS, LONS)

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import xarray as xr
import earthaccess
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def extract_transect(ds, var_name, target_lats, target_lons):
    """Extracts the nearest pixels to a defined transect line."""
    lat_vals = ds["latitude"].values
    lon_vals = ds["longitude"].values
    data_da = ds[var_name]

    rows, cols = [], []
    for lat, lon in zip(target_lats, target_lons):
        dist = np.abs(lat_vals - lat) + np.abs(lon_vals - lon)
        i, j = np.unravel_index(dist.argmin(), lat_vals.shape)
        rows.append(i)
        cols.append(j)

    selection = data_da.isel(
        number_of_lines=xr.DataArray(rows, dims="points"), 
        pixels_per_line=xr.DataArray(cols, dims="points")
    ).load()
    
    selection = selection.assign_coords(
        longitude=("points", [lon_vals[r, c] for r, c in zip(rows, cols)]),
        latitude=("points", [lat_vals[r, c] for r, c in zip(rows, cols)])
    )
    return selection

def plot_spectral_transect(ds_transect, file_id, date_str, output_path):
    """Generates and saves the spectral variation plot."""
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    cmap = plt.get_cmap('plasma') 
    
    fig, ax = plt.subplots(figsize=(12, 7))
    for i, p in enumerate(ds_transect.points):
        point_data = ds_transect.sel(points=p)
        color = cmap(norm(dist_degrees[i]))
        point_data.plot.line(ax=ax, x='wavelength_3d', marker='.', color=color, alpha=0.6, add_legend=False)
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('Distance (Degrees °)', rotation=270, labelpad=15)
    
    ax.set_title(f"Spectral Signatures: {date_str}", fontsize=14)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Surface Reflectance (rhos)")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    plt.savefig(os.path.join(output_path, f"{file_id}.png"), dpi=300)
    plt.close(fig)

def plot_transect_map(ds_transect, file_id, date_str, output_path):
    """Generates and saves a map showing the transect location."""
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)

    fig = plt.figure(figsize=(12, 9))
    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', edgecolor='dimgray')
    ax.add_feature(cfeature.OCEAN, facecolor='#e3f2fd')
    ax.add_feature(cfeature.COASTLINE, linewidth=1)
    
    ax.plot(lons, lats, color='black', linestyle='-', linewidth=1, alpha=0.3, transform=ccrs.PlateCarree(), zorder=1)
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    path = ax.scatter(lons, lats, c=dist_degrees, cmap='plasma', s=60, 
                      edgecolors='white', linewidth=0.5, transform=ccrs.PlateCarree(), zorder=2)

    ax.plot(lons[0], lats[0], marker='D', color='green', markersize=8, transform=ccrs.PlateCarree(), label='Start')
    ax.plot(lons[-1], lats[-1], marker='X', color='red', markersize=10, transform=ccrs.PlateCarree(), label='End')

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False

    cbar = plt.colorbar(path, ax=ax, orientation='vertical', shrink=0.6, pad=0.08)
    cbar.set_label('Distance (Degrees °)', rotation=270, labelpad=15)

    margin = 0.5
    ax.set_extent([lons.min() - margin, lons.max() + margin, lats.min() - margin, lats.max() + margin])
    
    plt.title(f"Transect Location: {file_id}\n(First available image: {date_str})", fontsize=14)
    plt.legend(loc='lower left')
    
    plt.savefig(os.path.join(output_path, f"{file_id}_map.png"), dpi=300)
    plt.close(fig)

def run_event_analysis(event_name, date_range, transect_lats, transect_lons):
    """Main workflow to process an event."""
    output_dir = os.path.join("output_reports", event_name)
    os.makedirs(output_dir, exist_ok=True)
    
    bbox = (min(transect_lons)-1, min(transect_lats)-1, max(transect_lons)+1, max(transect_lats)+1)
    
    print(f"Starting analysis for: {event_name}")
    results = earthaccess.search_data(short_name='PACE_OCI_L2_SFREFL', temporal=date_range, bounding_box=bbox)
    
    if not results:
        print("No files found.")
        return
        
    fileset = earthaccess.open(results)

    # Control para generar el mapa una sola vez
    map_generated = False

    for file_obj in fileset:
        try:
            # Abrir y combinar grupos del archivo HDF5/NetCDF
            dt = xr.open_datatree(file_obj, decode_timedelta=False, chunks={})
            ds = xr.merge(dt.to_dict().values())
            ds = ds.set_coords(("longitude", "latitude"))
            
            # Procesar el transecto
            rhos_transect = extract_transect(ds, 'rhos', transect_lats, transect_lons)
            
            # Extraer ID y Fecha para el nombre del archivo y títulos
            file_id = file_obj.full_name.split('.')[1]
            formatted_date = f"{file_id[:4]}-{file_id[4:6]}-{file_id[6:8]} {file_id[9:11]}:{file_id[11:13]}"
            
            # --- Lógica de Mapa Único ---
            if not map_generated:
                plot_transect_map(rhos_transect, event_name, formatted_date, output_dir)
                map_generated = True
                print(f"Map generated: {event_name}_map.png")
            
            # --- Gráfico Espectral ---
            plot_spectral_transect(rhos_transect, file_id, formatted_date, output_dir)
            
            print(f"Completed spectral plot: {file_id}")
        except Exception as e:
            print(f"Error processing {file_obj.full_name}: {e}")

if __name__ == "__main__":
    run_event_analysis(
        event_name = "Malacca_Strait_Bloom",
        date_range = ('2025-06-12', '2025-06-30'),
        transect_lats = np.linspace(1.9, 2.5, 20),
        transect_lons = np.linspace(102, 101.4, 20)
    )

Starting analysis for: Malacca_Strait_Bloom


QUEUEING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/24 [00:00<?, ?it/s]

Map generated: Malacca_Strait_Bloom_map.png
Completed spectral plot: 20250612T052037
Completed spectral plot: 20250612T052537


KeyboardInterrupt: 

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import xarray as xr
import earthaccess
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import io

# --- Processing Functions ---

def extract_transect(ds, var_name, target_lats, target_lons):
    """Extracts the nearest pixels to a defined transect line."""
    lat_vals = ds["latitude"].values
    lon_vals = ds["longitude"].values
    data_da = ds[var_name]

    rows, cols = [], []
    for lat, lon in zip(target_lats, target_lons):
        dist = np.abs(lat_vals - lat) + np.abs(lon_vals - lon)
        i, j = np.unravel_index(dist.argmin(), lat_vals.shape)
        rows.append(i)
        cols.append(j)

    # Load only the small subset into memory
    selection = data_da.isel(
        number_of_lines=xr.DataArray(rows, dims="points"), 
        pixels_per_line=xr.DataArray(cols, dims="points")
    ).load()
    
    selection = selection.assign_coords(
        longitude=("points", [lon_vals[r, c] for r, c in zip(rows, cols)]),
        latitude=("points", [lat_vals[r, c] for r, c in zip(rows, cols)])
    )
    return selection

def plot_spectral_transect(ds_transect, file_id, date_str, output_path):
    """Generates and saves the spectral variation plot."""
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    cmap = plt.get_cmap('plasma') 
    
    fig, ax = plt.subplots(figsize=(12, 7))
    for i, p in enumerate(ds_transect.points):
        point_data = ds_transect.sel(points=p)
        color = cmap(norm(dist_degrees[i]))
        point_data.plot.line(ax=ax, x='wavelength_3d', marker='.', color=color, alpha=0.6, add_legend=False)
    
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label('Distance (Degrees °)', rotation=270, labelpad=15)
    
    ax.set_title(f"Spectral Signatures: {date_str}", fontsize=14)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Surface Reflectance (rhos)")
    ax.grid(True, linestyle='--', alpha=0.3)
    
    plt.savefig(os.path.join(output_path, f"{file_id}.png"), dpi=300)
    plt.close(fig)

def plot_transect_map(ds_transect, event_name, date_str, output_path):
    """Generates and saves a map showing the transect location (once per event)."""
    lons = ds_transect.longitude.values
    lats = ds_transect.latitude.values
    dist_degrees = np.sqrt((lons - lons[0])**2 + (lats - lats[0])**2)

    fig = plt.figure(figsize=(12, 9))
    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', edgecolor='dimgray')
    ax.add_feature(cfeature.OCEAN, facecolor='#e3f2fd')
    ax.add_feature(cfeature.COASTLINE, linewidth=1)
    
    ax.plot(lons, lats, color='black', linestyle='-', linewidth=1, alpha=0.3, transform=ccrs.PlateCarree())
    
    norm = colors.Normalize(vmin=dist_degrees.min(), vmax=dist_degrees.max())
    path = ax.scatter(lons, lats, c=dist_degrees, cmap='plasma', s=60, 
                      edgecolors='white', linewidth=0.5, transform=ccrs.PlateCarree(), zorder=2)

    ax.plot(lons[0], lats[0], marker='D', color='green', markersize=8, transform=ccrs.PlateCarree(), label='Start')
    ax.plot(lons[-1], lats[-1], marker='X', color='red', markersize=10, transform=ccrs.PlateCarree(), label='End')

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = gl.right_labels = False

    cbar = plt.colorbar(path, ax=ax, orientation='vertical', shrink=0.6, pad=0.08)
    cbar.set_label('Distance (Degrees °)', rotation=270, labelpad=15)

    margin = 0.5
    ax.set_extent([lons.min() - margin, lons.max() + margin, lats.min() - margin, lats.max() + margin])
    
    plt.title(f"Transect Location: {event_name}\n(Reference: {date_str})", fontsize=14)
    plt.legend(loc='lower left')
    
    plt.savefig(os.path.join(output_path, f"{event_name}_map.png"), dpi=300)
    plt.close(fig)

def run_event_analysis(event_name, date_range, transect_lats, transect_lons):
    """Main workflow to process an event with memory purging."""
    output_dir = os.path.join("output_reports", event_name)
    os.makedirs(output_dir, exist_ok=True)
    
    bbox = (min(transect_lons)-1, min(transect_lats)-1, max(transect_lons)+1, max(transect_lats)+1)
    
    print(f"\n>>> Starting analysis for: {event_name}")
    results = earthaccess.search_data(short_name='PACE_OCI_L2_SFREFL', temporal=date_range, bounding_box=bbox)
    
    if not results:
        print(f"No files found for {event_name}")
        return
        
    fileset = earthaccess.open(results)
    map_generated = False

    for file_obj in fileset:
        try:
            # Use chunks={} for lazy loading (Dask)
            dt = xr.open_datatree(file_obj, decode_timedelta=False, chunks={})
            ds = xr.merge(dt.to_dict().values())
            ds = ds.set_coords(("longitude", "latitude"))
            
            rhos_transect = extract_transect(ds, 'rhos', transect_lats, transect_lons)
            
            file_id = file_obj.full_name.split('.')[1]
            formatted_date = f"{file_id[:4]}-{file_id[4:6]}-{file_id[6:8]} {file_id[9:11]}:{file_id[11:13]}"
            
            # Map generated only once per event
            if not map_generated:
                plot_transect_map(rhos_transect, event_name, formatted_date, output_dir)
                map_generated = True
                print(f"Event map generated: {event_name}_map.png")
            
            # Spectral plot per file
            plot_spectral_transect(rhos_transect, file_id, formatted_date, output_dir)
            print(f"Processed spectrum for: {file_id}")
            
            # --- MEMORY PURGE (Per File) ---
            ds.close()
            rhos_transect.close()
            del ds, dt, rhos_transect
            
        except Exception as e:
            print(f"Error processing {file_obj.full_name}: {e}")
        
        finally:
            # Clear Matplotlib backend and force garbage collection
            plt.close('all') 
            gc.collect()

    gc.collect()
    print(f"Memory purged after event: {event_name}")

# --- Main Block ---

if __name__ == "__main__":
    # 1. Load Event Data
    df_events = pd.read_csv('events.txt')

    # 2. Iterate through events
    for index, row in df_events.iterrows():
        # Generate transect points (20 points between start and end)
        lats = np.linspace(row['lat_start'], row['lat_end'], 20)
        lons = np.linspace(row['lon_start'], row['lon_end'], 20)
        
        run_event_analysis(
            event_name = row['event_id'],
            date_range = (row['date_ini'], row['date_end']),
            transect_lats = lats,
            transect_lons = lons
        )

    print("\n--- All events processed successfully ---")


>>> Starting analysis for: Egypt_Med_Spill


QUEUEING TASKS | :   0%|          | 0/22 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/22 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/22 [00:00<?, ?it/s]

Event map generated: Egypt_Med_Spill_map.png
Processed spectrum for: 20240710T095635
Processed spectrum for: 20240710T113454
Processed spectrum for: 20240711T103114
Processed spectrum for: 20240712T110552
Processed spectrum for: 20240713T100211
Processed spectrum for: 20240713T114030
Processed spectrum for: 20240714T103649
Processed spectrum for: 20240715T111126
Processed spectrum for: 20240716T100745
Processed spectrum for: 20240717T104222
Processed spectrum for: 20240718T093840
Processed spectrum for: 20240718T111659
Processed spectrum for: 20240718T111753
Processed spectrum for: 20240719T101317
Processed spectrum for: 20240720T104753
Processed spectrum for: 20240721T094411
Processed spectrum for: 20240721T112229
Processed spectrum for: 20240722T101846
Processed spectrum for: 20240723T105322
Processed spectrum for: 20240724T094939
Processed spectrum for: 20240724T112757
Processed spectrum for: 20240725T102414
Memory purged after event: Egypt_Med_Spill

>>> Starting analysis for: Chen

QUEUEING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/24 [00:00<?, ?it/s]

Event map generated: Malacca_Strait_Baseline_map.png
Processed spectrum for: 20250612T052037
Processed spectrum for: 20250612T052537
Processed spectrum for: 20250612T065858
Processed spectrum for: 20250613T055552
Processed spectrum for: 20250614T063108
Processed spectrum for: 20250615T052802
Processed spectrum for: 20250616T060314
Processed spectrum for: 20250619T061035
Processed spectrum for: 20250620T064549
Processed spectrum for: 20250621T054241
Processed spectrum for: 20250622T061755
Processed spectrum for: 20250623T051448
Processed spectrum for: 20250623T051948
Processed spectrum for: 20250623T065309
Processed spectrum for: 20250624T055001
Processed spectrum for: 20250625T062514
Processed spectrum for: 20250626T052206
Processed spectrum for: 20250626T052706
Processed spectrum for: 20250626T070027
Processed spectrum for: 20250627T055719
Processed spectrum for: 20250628T063231
Processed spectrum for: 20250629T052923
Processed spectrum for: 20250629T053423
Processed spectrum for: 202